# Seleção dinâmica dos ativos líquidos

Este notebook tem como objetivo construir o universo de ativos líquidos da B3 ao longo do tempo.

A seleção será feita por janelas temporais móveis. Em cada janela, serão mantidos todos os papéis que atenderem aos critérios mínimos de liquidez e cobertura de dados.

Os critérios usados serão:

- cobertura mínima de 80% dos pregões da janela;
- volume financeiro mediano positivo;
- quantidade mediana de negócios positiva.

### Importação das bibliotecas

Nesta etapa, importamos as bibliotecas necessárias para manipular dados, trabalhar com datas e organizar os caminhos dos arquivos do projeto.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

### Definição dos caminhos dos arquivos

Nesta etapa, definimos o arquivo de entrada e o arquivo de saída.

A entrada será a base de preços já combinada com informações cadastrais e setoriais, criada no notebook anterior.

A saída será a base histórica de ativos líquidos por janela.

In [3]:
arquivo_entrada = Path("../dados_tratados/dados_economatica_B3_com_setores.parquet")

arquivo_saida = Path("../dados_tratados/liquidez_historica.parquet")

print("Arquivo de entrada existe?", arquivo_entrada.exists())

arquivo_saida.parent.mkdir(parents=True, exist_ok=True)

Arquivo de entrada existe? True


### Carregamento da base de preços com setores

Nesta etapa, carregamos a base criada no notebook anterior.

Essa base contém os preços tratados e as informações cadastrais e setoriais necessárias para selecionar os ativos líquidos por janela.

In [4]:
df = pd.read_parquet(arquivo_entrada)

print("Tamanho da base:", df.shape)

display(df.head())

Tamanho da base: (1370766, 21)


,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ALLL11,2010-01-04,ALLL11<XBSP>,17.0387,16.5291,16.3192,17.0387,16.7789,"4,463.0000","39,094,265.0000","2,328,300.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
1,GVTT3,2010-01-05,GVTT3<XBSP>,55.6500,55.5400,55.5000,55.7000,55.5000,68.0000,"286,585,779.0000","5,163,600.0000",GVT Holding,ON,GVTT3,-,-,03420904000164,CANCELADA,-,-,-
2,ALLL11,2010-01-06,ALLL11<XBSP>,17.5584,18.0780,17.3985,18.5777,18.1780,"7,646.0000","78,531,111.0000","4,318,000.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
3,ALLL11,2010-01-07,ALLL11<XBSP>,17.3185,17.5883,17.0487,17.5883,17.3485,"4,952.0000","68,732,813.0000","3,958,300.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
4,AGIN3,2010-01-08,AGIN3<XBSP>,5.8100,5.6000,5.5900,5.8600,5.7400,"4,733.0000","45,060,702.0000","7,851,500.0000",Agra Incorp,ON,AGIN3,-,-,07698047000110,CANCELADA,-,-,-


### Validação das colunas necessárias

Antes de calcular a liquidez, verificamos se a base possui as colunas necessárias.

As principais colunas são:

- id_papel;
- ticker;
- data;
- setor;
- fechamento ajustado;
- volume financeiro;
- quantidade de negócios.

In [5]:
colunas_necessarias = [
    "id_papel",
    "ticker",
    "data",
    "setor",
    "fechamento_ajustado",
    "volume_financeiro",
    "q_negs"
]

colunas_faltantes = [
    col for col in colunas_necessarias
    if col not in df.columns
]

if colunas_faltantes:
    raise ValueError(f"Colunas faltantes: {colunas_faltantes}")

print("Todas as colunas necessárias estão presentes.")

Todas as colunas necessárias estão presentes.


### Preparação da base para cálculo de liquidez

Nesta etapa, garantimos que a coluna de data esteja no formato correto e ordenamos a base por data e identificador do papel.

A seleção de liquidez será feita por "id_papel", pois essa coluna usa o ISIN quando disponível e reduz o risco de tratar mudanças de ticker como ativos diferentes.

In [6]:
df_liq = df.copy()

df_liq["data"] = pd.to_datetime(df_liq["data"], errors="coerce")

df_liq = df_liq.sort_values(["data", "id_papel"]).reset_index(drop=True)

display(df_liq.head())

,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ALLL11,2010-01-04,ALLL11<XBSP>,17.0387,16.5291,16.3192,17.0387,16.7789,"4,463.0000","39,094,265.0000","2,328,300.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
1,ABCB4,2010-01-04,ABCB4<XBSP>,3.8282,3.7197,3.7197,3.8499,3.8189,407.0000,"2,479,611.0000","201,200.0000",Abc Brasil,PN,ABCB4,BRABCBACNPR4,BRABCBACNPR4,28195667000106,ATIVO,Financeiro,Intermediários financeiros,Bancos
2,ABEV3,2010-01-04,ABEV3<XBSP>,3.1813,3.0875,3.0565,3.1855,3.1401,94.0000,"4,938,379.0000","817,500.0000",Ambev S/A,ON,ABEV3,BRABEVACNOR1,BRABEVACNOR1,07526557000100,ATIVO,Consumo não cíclico,Bebidas,Cervejas e refrigerantes
3,AELP3,2010-01-04,AELP3<XBSP>,31.2875,31.2875,31.2875,31.2875,31.2875,2.0000,"29,400.0000",700.0000,AES Elpa,ON,AELP3,BRAELPACNOR2,BRAELPACNOR2,01917705000130,CANCELADA,Financeiro,Outros,Outros
4,AGEN33,2010-01-04,AGEN33<XBSP>,2.7900,2.9000,2.7500,2.9100,2.8000,"1,536.0000","12,736,907.0000","4,546,300.0000",Agrenco,ON,AGEN33,BRAGENBDR001,BRAGENBDR001,08943312000140,-,Consumo não cíclico,Comércio e distribuição,Alimentos


### Checagem do intervalo da base

Nesta etapa, verificamos o intervalo de datas disponível e a quantidade de papéis na base.

Essa checagem ajuda a confirmar se a base está coerente antes da criação das janelas temporais.

In [7]:
print("Data inicial:", df_liq["data"].min())
print("Data final:", df_liq["data"].max())
print("Quantidade de tickers:", df_liq["ticker"].nunique())
print("Quantidade de id_papel:", df_liq["id_papel"].nunique())
print("Quantidade de setores:", df_liq["setor"].nunique())

Data inicial: 2010-01-04 00:00:00
Data final: 2026-05-08 00:00:00
Quantidade de tickers: 755
Quantidade de id_papel: 746
Quantidade de setores: 12


### Parâmetros da seleção de liquidez

Nesta etapa, definimos os parâmetros que serão usados para selecionar os ativos líquidos.

A análise será feita apenas a partir de 2010.

A seleção será feita por janelas móveis de aproximadamente 1 ano de pregões. Em cada mês, será criada uma nova janela de formação usando apenas dados anteriores ou iguais à data final da janela.

Os critérios mínimos serão:

- cobertura mínima de 80% dos pregões da janela;
- volume financeiro mediano positivo;
- quantidade mediana de negócios positiva.

Não haverá limite máximo de ativos por janela. Todos os papéis que passarem nos critérios mínimos serão mantidos.

In [8]:
DATA_INICIAL = "2010-01-01"

JANELA_FORMACAO = 504

FREQ_REBALANCEAMENTO = "ME"

COBERTURA_MINIMA = 0.80

### Criação do calendário de pregões

Nesta etapa, criamos o calendário de pregões a partir das datas existentes na própria base.

Usar o calendário da base evita problemas com fins de semana, feriados e datas sem negociação.

In [9]:
calendario_pregoes = pd.Series(sorted(df_liq["data"].dropna().unique()))

print("Quantidade de pregões:", len(calendario_pregoes))
print("Primeiro pregão:", calendario_pregoes.iloc[0])
print("Último pregão:", calendario_pregoes.iloc[-1])

Quantidade de pregões: 4053
Primeiro pregão: 2010-01-04 00:00:00
Último pregão: 2026-05-08 00:00:00


### Definição das datas finais das janelas

A seleção de liquidez será feita mensalmente.

Para isso, usamos o último pregão disponível de cada mês como data final da janela.

Cada janela usa apenas dados anteriores ou iguais à sua data final, evitando uso de informação futura.

In [10]:
df_calendario = pd.DataFrame({"data": calendario_pregoes})

df_calendario = df_calendario.set_index("data")

datas_fim_janela = (
    df_calendario
    .resample(FREQ_REBALANCEAMENTO)
    .last()
    .dropna()
    .index
)

datas_fim_janela = datas_fim_janela[
    datas_fim_janela >= calendario_pregoes.iloc[JANELA_FORMACAO - 1]
]

print("Quantidade de janelas:", len(datas_fim_janela))
print("Primeira janela:", datas_fim_janela[0])
print("Última janela:", datas_fim_janela[-1])

Quantidade de janelas: 173
Primeira janela: 2012-01-31 00:00:00
Última janela: 2026-05-31 00:00:00


### Função para obter a janela de formação

Esta função recebe uma data final e retorna os dados dos últimos pregões da janela de formação (2 anos).

Ela garante que a seleção de liquidez use apenas informações disponíveis até a data final da janela e evitar bias.

In [11]:
def obter_janela_formacao(base, data_fim, tamanho_janela):
    datas_disponiveis = calendario_pregoes[calendario_pregoes <= data_fim]
    
    datas_janela = datas_disponiveis.iloc[-tamanho_janela:]
    
    data_inicio = datas_janela.iloc[0]
    
    janela = base[
        (base["data"] >= data_inicio) &
        (base["data"] <= data_fim)
    ].copy()
    
    return janela

### Função de seleção dos ativos líquidos

Nesta etapa, criamos a função que calcula a liquidez dos papéis dentro de uma janela.

A seleção será feita por "id_papel".

Para cada papel, calculamos:

- ticker mais recente na janela;
- ISIN;
- setor, subsetor e segmento;
- volume financeiro mediano;
- quantidade mediana de negócios;
- quantidade de observações com preço válido;
- cobertura de dados na janela.

Serão mantidos todos os papéis que tiverem:

- cobertura maior ou igual a 80%;
- volume financeiro mediano maior que zero;
- quantidade mediana de negócios maior que zero.

O ranking de liquidez será calculado apenas para ordenação e análise descritiva, não como critério de exclusão.

In [12]:
def selecionar_ativos_liquidos(janela):
    total_pregoes_janela = janela["data"].nunique()
    
    liquidez = (
        janela
        .groupby("id_papel")
        .agg(
            ticker=("ticker", "last"),
            isin=("isin", "last"),
            nome=("nome", "last"),
            classe=("classe", "last"),
            situacao_cvm=("situacao_cvm", "last"),
            setor=("setor", "last"),
            subsetor=("subsetor", "last"),
            segmento=("segmento", "last"),
            volume_mediano=("volume_financeiro", "median"),
            negocios_mediano=("q_negs", "median"),
            obs_preco=("fechamento_ajustado", "count"),
            primeira_data=("data", "min"),
            ultima_data=("data", "max")
        )
        .reset_index()
    )
    
    liquidez["cobertura"] = liquidez["obs_preco"] / total_pregoes_janela
    
    liquidez = liquidez[liquidez["cobertura"] >= COBERTURA_MINIMA].copy()
    
    liquidez = liquidez[liquidez["volume_mediano"] > 0].copy()
    
    liquidez = liquidez[liquidez["negocios_mediano"] > 0].copy()
    
    liquidez["rank_volume"] = liquidez["volume_mediano"].rank(
        ascending=False,
        method="min"
    )
    
    liquidez["rank_negocios"] = liquidez["negocios_mediano"].rank(
        ascending=False,
        method="min"
    )
    
    liquidez["score_liquidez"] = (
        liquidez["rank_volume"] + liquidez["rank_negocios"]
    )
    
    liquidez = liquidez.sort_values(
        ["score_liquidez", "rank_volume", "rank_negocios"]
    ).reset_index(drop=True)
    
    return liquidez

### Teste da seleção de liquidez em uma janela

Antes de aplicar a seleção para todas as janelas, testamos a função em uma única janela.

In [13]:
data_teste = datas_fim_janela[0]

janela_teste = obter_janela_formacao(
    base=df_liq,
    data_fim=data_teste,
    tamanho_janela=JANELA_FORMACAO
)

liquidez_teste = selecionar_ativos_liquidos(janela_teste)

print("Data final da janela:", data_teste)
print("Pregões na janela:", janela_teste["data"].nunique())
print("Papéis líquidos selecionados:", liquidez_teste["id_papel"].nunique())
print("Setores representados:", liquidez_teste["setor"].nunique())
print("Papéis sem setor:", liquidez_teste["setor"].isna().sum())

display(liquidez_teste.head(30))

Data final da janela: 2012-01-31 00:00:00
Pregões na janela: 504
Papéis líquidos selecionados: 262
Setores representados: 12
Papéis sem setor: 0


,id_papel,ticker,isin,nome,classe,situacao_cvm,setor,subsetor,segmento,volume_mediano,negocios_mediano,obs_preco,primeira_data,ultima_data,cobertura,rank_volume,rank_negocios,score_liquidez
0,BRVALEACNPA3,VALE5,BRVALEACNPA3,Vale,PNA,ATIVO,Materiais básicos,Mineração,Minerais metálicos,"662,657,009.5000","19,193.5000",504,2010-01-21,2012-01-31,1.0000,1.0000,2.0000,3.0000
1,BRPETRACNPR6,PETR4,BRPETRACNPR6,Petrobras,PN,ATIVO,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"500,959,699.0000","19,891.5000",504,2010-01-21,2012-01-31,1.0000,2.0000,1.0000,3.0000
2,BROGXPACNOR3,OGXP3,BROGXPACNOR3,OGX Petroleo,ON,CANCELADA,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"259,763,936.5000","12,576.5000",504,2010-01-21,2012-01-31,1.0000,3.0000,3.0000,6.0000
3,BRITUBACNPR1,ITUB4,BRITUBACNPR1,ItauUnibanco,PN,ATIVO,Financeiro,Intermediários financeiros,Bancos,"219,055,962.5000","10,878.5000",504,2010-01-21,2012-01-31,1.0000,4.0000,4.0000,8.0000
4,BRB3SAACNOR6,B3SA3,BRB3SAACNOR6,B3,ON,ATIVO,Financeiro,Serviços financeiros diversos,Serviços financeiros diversos,"136,037,162.0000","10,841.5000",504,2010-01-21,2012-01-31,1.0000,8.0000,5.0000,13.0000
5,BRBBDCACNPR8,BBDC4,BRBBDCACNPR8,Bradesco,PN,ATIVO,Financeiro,Intermediários financeiros,Bancos,"152,271,531.0000","8,585.0000",504,2010-01-21,2012-01-31,1.0000,6.0000,9.0000,15.0000
6,BRBBASACNOR3,BBAS3,BRBBASACNOR3,Brasil,ON,ATIVO,Financeiro,Intermediários financeiros,Bancos,"130,177,646.5000","8,925.0000",504,2010-01-21,2012-01-31,1.0000,9.0000,8.0000,17.0000
7,BRGGBRACNPR8,GGBR4,BRGGBRACNPR8,Gerdau,PN,ATIVO,Materiais básicos,Siderurgia e metalurgia,Siderurgia,"118,063,737.0000","9,309.0000",504,2010-01-21,2012-01-31,1.0000,10.0000,7.0000,17.0000
8,BRPDGRACNOR8,PDGR3,BRPDGRACNOR8,PDG Realt,ON,ATIVO,Consumo cíclico,Construção civil,Incorporações,"85,308,504.0000","10,361.5000",504,2010-01-21,2012-01-31,1.0000,12.0000,6.0000,18.0000
9,BRPETRACNOR9,PETR3,BRPETRACNOR9,Petrobras,ON,ATIVO,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"137,090,268.5000","6,889.5000",504,2010-01-21,2012-01-31,1.0000,7.0000,12.0000,19.0000


### Execução da seleção de liquidez em todas as janelas

Nesta etapa, aplicamos a seleção de liquidez para todas as janelas temporais.

Para cada janela:

1. pegamos os dados da janela de formação;
2. calculamos as métricas de liquidez;
3. mantemos os papéis que passam nos critérios mínimos;
4. salvamos o resultado da janela.

O resultado será a base histórica de ativos líquidos.

In [14]:
resultados_liquidez = []

for data_fim in datas_fim_janela:
    janela = obter_janela_formacao(
        base=df_liq,
        data_fim=data_fim,
        tamanho_janela=JANELA_FORMACAO
    )
    
    liquidez_janela = selecionar_ativos_liquidos(janela)
    
    liquidez_janela["data_fim_janela"] = data_fim
    liquidez_janela["pregoes_janela"] = janela["data"].nunique()
    
    resultados_liquidez.append(liquidez_janela)

liquidez_historica = pd.concat(resultados_liquidez, ignore_index=True)

print("Tamanho da base de liquidez histórica:", liquidez_historica.shape)

display(liquidez_historica.head())

Tamanho da base de liquidez histórica: (48752, 20)


,id_papel,ticker,isin,nome,classe,situacao_cvm,setor,subsetor,segmento,volume_mediano,negocios_mediano,obs_preco,primeira_data,ultima_data,cobertura,rank_volume,rank_negocios,score_liquidez,data_fim_janela,pregoes_janela
0,BRVALEACNPA3,VALE5,BRVALEACNPA3,Vale,PNA,ATIVO,Materiais básicos,Mineração,Minerais metálicos,"662,657,009.5000","19,193.5000",504,2010-01-21,2012-01-31,1.0000,1.0000,2.0000,3.0000,2012-01-31,504
1,BRPETRACNPR6,PETR4,BRPETRACNPR6,Petrobras,PN,ATIVO,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"500,959,699.0000","19,891.5000",504,2010-01-21,2012-01-31,1.0000,2.0000,1.0000,3.0000,2012-01-31,504
2,BROGXPACNOR3,OGXP3,BROGXPACNOR3,OGX Petroleo,ON,CANCELADA,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"259,763,936.5000","12,576.5000",504,2010-01-21,2012-01-31,1.0000,3.0000,3.0000,6.0000,2012-01-31,504
3,BRITUBACNPR1,ITUB4,BRITUBACNPR1,ItauUnibanco,PN,ATIVO,Financeiro,Intermediários financeiros,Bancos,"219,055,962.5000","10,878.5000",504,2010-01-21,2012-01-31,1.0000,4.0000,4.0000,8.0000,2012-01-31,504
4,BRB3SAACNOR6,B3SA3,BRB3SAACNOR6,B3,ON,ATIVO,Financeiro,Serviços financeiros diversos,Serviços financeiros diversos,"136,037,162.0000","10,841.5000",504,2010-01-21,2012-01-31,1.0000,8.0000,5.0000,13.0000,2012-01-31,504


### Organização da base de liquidez histórica

Nesta etapa, organizamos as colunas da base final de liquidez e definimos um índice composto.

Cada linha da base representa um papel líquido em uma determinada janela temporal.

Como o mesmo ativo aparece em várias janelas, o ticker sozinho não identifica uma linha única. Além disso, o ticker pode mudar ao longo do tempo.

Por isso, usamos um índice composto por "data_fim_janela" e "id_papel".

A coluna "data_fim_janela" indica a data em que o universo líquido foi calculado. A coluna "id_papel" identifica o papel, usando o ISIN quando disponível e o ticker quando o ISIN está ausente.

In [15]:
colunas_liquidez = [
    "data_fim_janela",
    "id_papel",
    "ticker",
    "isin",
    "nome",
    "classe",
    "situacao_cvm",
    "setor",
    "subsetor",
    "segmento",
    "volume_mediano",
    "negocios_mediano",
    "obs_preco",
    "cobertura",
    "rank_volume",
    "rank_negocios",
    "score_liquidez",
    "primeira_data",
    "ultima_data",
    "pregoes_janela"
]

liquidez_historica = liquidez_historica[colunas_liquidez].copy()

liquidez_historica = liquidez_historica.sort_values(
    ["data_fim_janela", "score_liquidez"]
).reset_index(drop=True)

liquidez_historica_indexada = liquidez_historica.set_index(
    ["data_fim_janela", "id_papel"]
)

display(liquidez_historica_indexada.head(20))

ticker          isin          nome classe  \
data_fim_janela id_papel                                                 
2012-01-31      BRVALEACNPA3  VALE5  BRVALEACNPA3          Vale    PNA   
                BRPETRACNPR6  PETR4  BRPETRACNPR6     Petrobras     PN   
                BROGXPACNOR3  OGXP3  BROGXPACNOR3  OGX Petroleo     ON   
                BRITUBACNPR1  ITUB4  BRITUBACNPR1  ItauUnibanco     PN   
                BRB3SAACNOR6  B3SA3  BRB3SAACNOR6            B3     ON   
                BRBBDCACNPR8  BBDC4  BRBBDCACNPR8      Bradesco     PN   
                BRBBASACNOR3  BBAS3  BRBBASACNOR3        Brasil     ON   
                BRGGBRACNPR8  GGBR4  BRGGBRACNPR8        Gerdau     PN   
                BRPDGRACNOR8  PDGR3  BRPDGRACNOR8     PDG Realt     ON   
                BRPETRACNOR9  PETR3  BRPETRACNOR9     Petrobras     ON   
                BRVALEACNOR0  VALE3  BRVALEACNOR0          Vale     ON   
                BRUSIMACNPA6  USIM5  BRUSIMACNPA6      Usiminas    PNA   
                BRITSAACNPR7  ITSA4  BRITSAACNPR7        Itausa     PN   
                BRCYREACNOR7  CYRE3  BRCYREACNOR7  Cyrela Realt     ON   
                BRCSNAACNOR6  CSNA3  BRCSNAACNOR6  Sid Nacional     ON   
                BRGFSAACNOR3  GFSA3  BRGFSAACNOR3        Gafisa     ON   
                BRMRVEACNOR2  MRVE3  BRMRVEACNOR2           MRV     ON   
                BRCIELACNOR3  CIEL3  BRCIELACNOR3         Cielo     ON   
                BRBRFSACNOR8  BRFS3  BRBRFSACNOR8        BRF SA     ON   
                BRRDCDACNOR3  RDCD3  BRRDCDACNOR3      Redecard     ON   

                             situacao_cvm                           setor  \
data_fim_janela id_papel                                                    
2012-01-31      BRVALEACNPA3        ATIVO               Materiais básicos   
                BRPETRACNPR6        ATIVO  Petróleo gás e biocombustíveis   
                BROGXPACNOR3    CANCELADA  Petróleo gás e biocombustíveis   
                BRITUBACNPR1        ATIVO                      Financeiro   
                BRB3SAACNOR6        ATIVO                      Financeiro   
                BRBBDCACNPR8        ATIVO                      Financeiro   
                BRBBASACNOR3        ATIVO                      Financeiro   
                BRGGBRACNPR8        ATIVO               Materiais básicos   
                BRPDGRACNOR8        ATIVO                 Consumo cíclico   
                BRPETRACNOR9        ATIVO  Petróleo gás e biocombustíveis   
                BRVALEACNOR0        ATIVO               Materiais básicos   
                BRUSIMACNPA6        ATIVO               Materiais básicos   
                BRITSAACNPR7        ATIVO                      Financeiro   
                BRCYREACNOR7        ATIVO                 Consumo cíclico   
                BRCSNAACNOR6        ATIVO               Materiais básicos   
                BRGFSAACNOR3        ATIVO                 Consumo cíclico   
                BRMRVEACNOR2        ATIVO                 Consumo cíclico   
                BRCIELACNOR3        ATIVO                      Financeiro   
                BRBRFSACNOR8        ATIVO             Consumo não cíclico   
                BRRDCDACNOR3    CANCELADA                               -   

                                                    subsetor  \
data_fim_janela id_papel                                       
2012-01-31      BRVALEACNPA3                       Mineração   
                BRPETRACNPR6  Petróleo gás e biocombustíveis   
                BROGXPACNOR3  Petróleo gás e biocombustíveis   
                BRITUBACNPR1      Intermediários financeiros   
                BRB3SAACNOR6   Serviços financeiros diversos   
                BRBBDCACNPR8      Intermediários financeiros   
                BRBBASACNOR3      Intermediários financeiros   
                BRGGBRACNPR8         Siderurgia e metalurgia   
                BRPDGRACNOR8                Construção civil   
 

### Checagem da quantidade de ativos líquidos por janela

Verificamos quantos papéis foram selecionados em cada janela.

In [16]:
ativos_por_janela = (
    liquidez_historica
    .groupby("data_fim_janela")["id_papel"]
    .nunique()
    .reset_index(name="qtd_papeis_liquidos")
)

display(ativos_por_janela.describe())

display(ativos_por_janela.head())

display(ativos_por_janela.tail())

,data_fim_janela,qtd_papeis_liquidos
count,173,173.0000
mean,2019-03-31 18:18:43.699422,281.8035
min,2012-01-31 00:00:00,230.0000
25%,2015-08-31 00:00:00,251.0000
50%,2019-03-31 00:00:00,264.0000
75%,2022-10-31 00:00:00,324.0000
max,2026-05-31 00:00:00,355.0000
std,NaN,38.8781


,data_fim_janela,qtd_papeis_liquidos
0,2012-01-31,262
1,2012-02-29,261
2,2012-03-31,262
3,2012-04-30,261
4,2012-05-31,259


,data_fim_janela,qtd_papeis_liquidos
168,2026-01-31,325
169,2026-02-28,325
170,2026-03-31,324
171,2026-04-30,322
172,2026-05-31,320


### Checagem de ativos líquidos por setor

Verificamos quantos papéis líquidos existem em cada setor e em cada janela.

Essa checagem é importante porque, no próximo notebook, os pares serão formados apenas entre ativos do mesmo setor.

Para formar pares dentro de um setor, é necessário ter pelo menos dois papéis líquidos na mesma janela.

In [17]:
ativos_por_setor_janela = (
    liquidez_historica
    .dropna(subset=["setor"])
    .groupby(["data_fim_janela", "setor"])["id_papel"]
    .nunique()
    .reset_index(name="qtd_papeis_setor")
)

display(ativos_por_setor_janela.head(30))

display(ativos_por_setor_janela["qtd_papeis_setor"].describe())

,data_fim_janela,setor,qtd_papeis_setor
0,2012-01-31,-,29
1,2012-01-31,Bens industriais,40
2,2012-01-31,Comunicações,7
3,2012-01-31,Consumo cíclico,50
4,2012-01-31,Consumo não cíclico,16
5,2012-01-31,Financeiro,44
6,2012-01-31,Materiais básicos,30
7,2012-01-31,Outros,3
8,2012-01-31,Petróleo gás e biocombustíveis,9
9,2012-01-31,Saúde,7


count   1,969.0000
mean       24.7598
std        19.3508
min         1.0000
25%         9.0000
50%        18.0000
75%        41.0000
max        81.0000
Name: qtd_papeis_setor, dtype: float64

### Checagem de setores com possibilidade de formação de pares

Verificamos quantos setores possuem pelo menos dois papéis líquidos em cada janela.

Esses são os setores que poderão gerar pares no próximo notebook.

In [18]:
setores_com_pares_possiveis = (
    ativos_por_setor_janela
    .assign(pode_formar_par=ativos_por_setor_janela["qtd_papeis_setor"] >= 2)
    .groupby("data_fim_janela")["pode_formar_par"]
    .sum()
    .reset_index(name="setores_com_pelo_menos_2_ativos")
)

display(setores_com_pares_possiveis.describe())

display(setores_com_pares_possiveis.head())

display(setores_com_pares_possiveis.tail())

,data_fim_janela,setores_com_pelo_menos_2_ativos
count,173,173.0000
mean,2019-03-31 18:18:43.699422,11.0809
min,2012-01-31 00:00:00,10.0000
25%,2015-08-31 00:00:00,11.0000
50%,2019-03-31 00:00:00,11.0000
75%,2022-10-31 00:00:00,12.0000
max,2026-05-31 00:00:00,12.0000
std,NaN,0.7427


,data_fim_janela,setores_com_pelo_menos_2_ativos
0,2012-01-31,12
1,2012-02-29,12
2,2012-03-31,12
3,2012-04-30,12
4,2012-05-31,12


,data_fim_janela,setores_com_pelo_menos_2_ativos
168,2026-01-31,10
169,2026-02-28,10
170,2026-03-31,10
171,2026-04-30,10
172,2026-05-31,10


### Checagem de ativos sem setor

verificamos se existem papéis líquidos sem setor identificado.

Esses ativos não serão removidos agora. Porém, no próximo notebook, eles não poderão ser usados na formação de pares setoriais.

In [19]:
sem_setor_por_janela = (
    liquidez_historica
    .assign(sem_setor=liquidez_historica["setor"].isna())
    .groupby("data_fim_janela")["sem_setor"]
    .sum()
    .reset_index(name="ativos_sem_setor")
)

display(sem_setor_por_janela.describe())

display(sem_setor_por_janela.head())

display(sem_setor_por_janela.tail())

,data_fim_janela,ativos_sem_setor
count,173,173.0000
mean,2019-03-31 18:18:43.699422,0.0000
min,2012-01-31 00:00:00,0.0000
25%,2015-08-31 00:00:00,0.0000
50%,2019-03-31 00:00:00,0.0000
75%,2022-10-31 00:00:00,0.0000
max,2026-05-31 00:00:00,0.0000
std,NaN,0.0000


,data_fim_janela,ativos_sem_setor
0,2012-01-31,0
1,2012-02-29,0
2,2012-03-31,0
3,2012-04-30,0
4,2012-05-31,0


,data_fim_janela,ativos_sem_setor
168,2026-01-31,0
169,2026-02-28,0
170,2026-03-31,0
171,2026-04-30,0
172,2026-05-31,0


### Salvamento da base de liquidez histórica

Por fim, salvamos a base de liquidez histórica em formato Parquet.

In [20]:
liquidez_historica.to_parquet(arquivo_saida, index=False)

print("Base de liquidez histórica salva em:", arquivo_saida)

Base de liquidez histórica salva em: ..\dados_tratados\liquidez_historica.parquet
